# BME i9400 — Meeting 5
## Studio: PCA of a Gene Expression Matrix

**Fall 2026 · Wednesday, September 16**

Today you implement the method from Meeting 4 on the tumour expression matrix: centering, the
singular value decomposition, the scree plot, the projection, and a measurement of what the
compression costs.

**Agenda**

1. Loading and inspecting the matrix
2. Centering
3. The singular value decomposition
4. Choosing how many components to keep
5. Projecting and plotting
6. What does PC1 actually measure?
7. What the compression costs
8. Submitting your work

Section 1 is done for you — just read and run it. Sections 2 to 7 contain tasks marked `TODO` that
you are asked to complete. Each task ends with checks that must pass before you move on.

---
## 1 — Loading and inspecting the matrix

The data is a subset of The Cancer Genome Atlas: 801 tumour samples, each with expression levels for
the 400 genes that vary most across the collection, plus a label giving the cancer type.

> **RNA-seq** measures how strongly each gene is being expressed in a tissue sample. A higher number
> means more of that gene's RNA was present.

The five cancer types are BRCA (breast), COAD (colon), KIRC (kidney), LUAD (lung), and PRAD
(prostate).

**The labels are used only for colouring the plots.** Every calculation in sections 2 to 5 sees the
expression matrix alone.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 12, "figure.dpi": 110})

URL = "https://raw.githubusercontent.com/dmochow/BME-i9400-2026/main/data/tcga_rnaseq_subset.csv"
df  = pd.read_csv(URL)

y = df["cancer_type"].values                    # labels, for plotting only
X = df.drop(columns="cancer_type").values       # the expression matrix
genes = df.drop(columns="cancer_type").columns  # gene names, for section 6

print(f"X has shape {X.shape}  ->  n = {X.shape[0]} samples, d = {X.shape[1]} genes")
print("cancer types:", ", ".join(f"{c} ({(y==c).sum()})" for c in sorted(set(y))))
print(f"\nexpression values range from {X.min():.2f} to {X.max():.2f}")
df.iloc[:4, :5]

---
## 2 — Centering

PCA is defined on centered data: every column must have mean zero. Section 4 of Meeting 4:

$$\tilde{X} = X - \bar{x}$$

where $\bar{x}$ is the vector of the 400 column means.

> `X.mean(axis=0)` averages **down the columns**, giving one number per gene — a vector of length
> $d$. `X.mean(axis=1)` would average across the row, giving one number per sample, which is not
> what we want.
>
> NumPy **broadcasting** lets you subtract a length-400 vector from an 801-by-400 matrix: the vector
> is applied to every row automatically.

In [ ]:
# --- TODO: center the matrix ---
gene_means = None      # the mean of each gene, a vector of length 400
Xc         = None      # the centered matrix, same shape as X

assert gene_means is not None and Xc is not None, "Fill in both TODOs above."
assert gene_means.shape == (X.shape[1],), f"gene_means should have shape ({X.shape[1]},)"
assert Xc.shape == X.shape, "Xc should be the same shape as X."
assert np.abs(Xc.mean(axis=0)).max() < 1e-9, "Columns of Xc are not mean-zero."

print(f"largest remaining column mean: {np.abs(Xc.mean(axis=0)).max():.2e}   (should be ~0)")
print(f"a few gene means before centering: {np.round(gene_means[:4], 3)}")

---
## 3 — The singular value decomposition

$$\tilde{X} = U S V^{\top}$$

`np.linalg.svd(Xc, full_matrices=False)` returns three arrays: `U`, `S`, and `Vt`.

> `Vt` is $V^{\top}$ — already transposed. So **the components are the rows of `Vt`**, not its
> columns: `Vt[0]` is the first principal component, a vector of 400 gene weights.
>
> `S` is returned as a one-dimensional array of singular values, not as a diagonal matrix.

From Meeting 4, the variance along component $j$ is $\lambda_j = s_j^2/(n-1)$, so the fraction of
total variance it explains is

$$\frac{s_j^2}{s_1^2 + s_2^2 + \cdots}$$

In [ ]:
# --- TODO: decompose the centered matrix ---
U, S, Vt = None, None, None       # use np.linalg.svd with full_matrices=False

assert U is not None, "Fill in the TODO above."
assert Vt.shape[1] == X.shape[1], "Each row of Vt should have one weight per gene."
assert np.allclose(np.linalg.norm(Vt[0]), 1.0), "Components should be unit vectors."
assert np.abs(Vt[0] @ Vt[1]) < 1e-9, "Components should be perpendicular."

# --- TODO: fraction of total variance explained by each component ---
var_frac = None                   # array the same length as S, summing to 1

assert var_frac is not None, "Fill in var_frac."
assert np.isclose(var_frac.sum(), 1.0), "var_frac should sum to 1."
assert np.all(np.diff(var_frac) <= 1e-12), "var_frac should be in decreasing order."

print("singular values, first 5 :", np.round(S[:5], 1))
print("variance explained, first 5:", np.round(100*var_frac[:5], 1), "%")
print(f"\nPC1 + PC2 retain {100*var_frac[:2].sum():.1f}% of the total variance")

---
## 4 — Choosing how many components to keep

The scree plot shows variance explained against component number. There is no rule that fixes the
answer; two common approaches are to look for the point where the plot flattens, and to keep however
many components are needed to reach a chosen cumulative total.

In [ ]:
# --- TODO: how many components are needed to reach 90% of the total variance? ---
# np.cumsum gives running totals; np.searchsorted or a comparison plus .sum() will finish it.
cumulative = None      # running total of var_frac
n_for_90   = None      # smallest number of components with cumulative variance >= 0.90

assert cumulative is not None and n_for_90 is not None, "Fill in both TODOs."
assert cumulative[-1] > 0.999, "cumulative should end at 1."
assert cumulative[n_for_90 - 1] >= 0.90 and cumulative[n_for_90 - 2] < 0.90, \
    "n_for_90 is not the smallest count reaching 90%."

print(f"components needed for 90% of the variance: {n_for_90} of {len(S)}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].bar(np.arange(1, 21), 100*var_frac[:20], color="#5b3a8e")
axes[0].set_xlabel("component"); axes[0].set_ylabel("variance explained (%)")
axes[0].set_title("Scree plot: first 20 components")

axes[1].plot(np.arange(1, len(S)+1), 100*cumulative, lw=2, color="#5b3a8e")
axes[1].axhline(90, ls="--", color="#c1272d")
axes[1].axvline(n_for_90, ls="--", color="#c1272d")
axes[1].set_xlabel("number of components kept"); axes[1].set_ylabel("cumulative variance (%)")
axes[1].set_title(f"90% reached at {n_for_90} components")
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 5 — Projecting and plotting

Projecting sample $x$ onto component $v$ is the dot product $x \cdot v$, from section 3 of Meeting 4.
Doing it for every sample and the first two components at once is a single matrix product:

$$Z = \tilde{X}\, V_{[:2]}^{\top} \qquad Z \in \mathbb{R}^{n \times 2}$$

> `Vt[:2]` is a 2-by-400 array holding the first two components. Its transpose is 400-by-2, so
> `Xc @ Vt[:2].T` has shape 801-by-2 — two coordinates per tumour.

In [ ]:
# --- TODO: project every sample onto the first two components ---
Z = None       # shape (801, 2)

assert Z is not None, "Fill in Z."
assert Z.shape == (X.shape[0], 2), f"Z should have shape ({X.shape[0]}, 2), got {Z.shape}"
assert np.isclose(Z[:, 0].var(ddof=1), S[0]**2/(len(X)-1), rtol=1e-6), \
    "Variance along PC1 should equal its eigenvalue."

fig, ax = plt.subplots(figsize=(8.5, 6.5))
for c, col in zip(sorted(set(y)), ["#c1272d", "#2b6cb0", "#e8a33d", "#2f8f5b", "#7d5ba6"]):
    m = y == c
    ax.scatter(Z[m, 0], Z[m, 1], s=18, alpha=0.75, color=col, label=c)
ax.set_xlabel(f"PC1  ({100*var_frac[0]:.1f}% of variance)")
ax.set_ylabel(f"PC2  ({100*var_frac[1]:.1f}%)")
ax.set_title("801 tumours, 400 genes, drawn in two dimensions")
ax.legend(title="cancer type"); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

---
## 6 — What does PC1 actually measure?

A principal component is a vector of 400 numbers, one weight per gene. Genes with large weights
contribute most to that component, so inspecting them says what the component is responding to.

This is a property of PCA worth noting: because each component is a weighted sum of the original
features, it can be read. Most of the embeddings later in this course cannot be.

In [ ]:
# --- TODO: find the 5 genes with the largest absolute weight in PC1 ---
# np.argsort sorts ascending and returns indices; np.abs gives magnitudes.
pc1 = Vt[0]
top5_idx = None        # indices of the 5 largest |weight| in pc1, largest first

assert top5_idx is not None, "Fill in top5_idx."
assert len(top5_idx) == 5, "Give exactly 5 indices."
assert np.abs(pc1[top5_idx[0]]) >= np.abs(pc1[top5_idx[-1]]), "Order them largest first."

print("Genes contributing most to PC1:\n")
for rank, j in enumerate(top5_idx, 1):
    print(f"  {rank}. {genes[j]:>12s}   weight {pc1[j]:+.4f}")

print(f"\nThese 5 of 400 genes carry "
      f"{100*np.sum(pc1[top5_idx]**2):.1f}% of PC1's total weight.")

---
## 7 — What the compression costs

Section 5 replaced 400 numbers per tumour with 2. The question that matters is how much was lost.

A nearest-neighbour classifier gives one answer: train it on the full 400-gene representation, train
it again on the 2-dimensional one, and compare accuracy.

> **Cross-validation** splits the data into folds, trains on all but one, tests on the held-out fold,
> and repeats. `cross_val_score` returns one accuracy per fold. Meeting 11 covers why this is done
> and where it goes wrong; today it is just a way to get a number.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

# --- TODO: 5-fold accuracy of a 5-nearest-neighbour classifier on each representation ---
# cross_val_score(KNeighborsClassifier(5), FEATURES, y, cv=5).mean()
acc_full = None      # using all 400 genes  (X)
acc_2pc  = None      # using the 2 principal components  (Z)

assert acc_full is not None and acc_2pc is not None, "Fill in both."

print(f"accuracy on all 400 genes      : {acc_full:.3f}")
print(f"accuracy on 2 principal components: {acc_2pc:.3f}")
print(f"\ndifference: {acc_full - acc_2pc:.3f}")
print(f"features used: {X.shape[1]} -> 2   ({100*2/X.shape[1]:.1f}% of the original)")

Two hundredfold fewer numbers per sample, for a small drop in accuracy. That is the case for PCA on
this dataset.

It is not a general result. PCA orders directions by variance, and variance is not the same thing as
usefulness for a particular task. Here the largest-variance directions happen to be the ones that
distinguish tissue of origin. On a different question — separating tumour grades within one cancer
type, say — the informative direction might carry very little variance and be discarded.

---
## 8 — Submitting your work

**Due at 11:59 PM EST on Wednesday, September 16.**

1. `Runtime ▸ Restart session and run all`. Every cell must run without error, and every check must
   pass.
2. `File ▸ Download ▸ Download .ipynb`.
3. Rename the file to `YOUR-GITHUB-USERNAME_meeting05.ipynb` and move it into the `checkins/` folder
   of your clone.
4. Commit in GitHub Desktop, push, and open a pull request.

### Two short questions

Answer in the cell below before you submit.

In [ ]:
# One or two sentences each.

# 1. No cancer type labels were used in sections 2 to 5, yet the five types separate in the PC1-PC2
#    plot. What does that tell you about the expression matrix?
Q1 = ""

# 2. Section 4 found that reaching 90% of the variance takes many more than 2 components, but
#    section 7 showed that 2 components already classify almost as well as all 400 genes.
#    How can both be true?
Q2 = ""

for name, ans in [("Q1", Q1), ("Q2", Q2)]:
    assert len(ans.strip()) >= 30, f"{name} looks empty or very short."
print("Answers recorded.")